## PART I:  fitting the signal, bakground

In [ ]:
import sys,os
import numpy as np
import scipy
from scipy.integrate import quad
from scipy.stats import kde
from scipy.stats import chi2 as spchi2
from scipy.optimize import minimize,leastsq
import matplotlib
import matplotlib.pyplot as py
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import rc
rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc('text', usetex=True)  
%matplotlib inline  
%config InlineBackend.figure_format = 'retina'
from tools import tex,save, load,fill_between,plot_hist
from tools import EVENTS,VEC4

## The background

Retrieve simulated background and construct lepton-pair invariant mass

In [ ]:
path = 'samples/bkg.lhe.gz'
bkg  = EVENTS(path)
bkg.D['m'] = []
for i in range(bkg.nevents):
    particles = bkg.EVENTS[i]
    for p in particles:
        if p['pid']==-13: mub=p['mom']
        elif p['pid']==13: mu=p['mom']
    z = mu + mub
    bkg.D['m'].append((z*z)**0.5) 

construct observable as $\frac{1}{N_{tot}}~~\frac{dN}{dm}$

In [ ]:
lower_binning = 40
upper_binning = 200
number_of_bins = 100
N,E = np.histogram(bkg.D['m'],bins=number_of_bins,range=(lower_binning,upper_binning))

M = 0.5*(E[:-1]+E[1:])
NTOT = np.sum(N)
SNSQ = np.sum(np.sqrt(N))
OBS  = N/float(NTOT)
ERR  = np.zeros(OBS.size)
for k in range(len(ERR)):
        ERR[k] = np.sqrt(N[k])/float(NTOT)
                      
ax = py.subplot(111)
ax.errorbar(M,OBS,yerr=ERR,fmt='g.')
ax.set_ylabel(tex(r'bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

model the bkg and make fit

In [ ]:
rs = 1300.0
model_bkg = lambda p,m: p[0]/m**(p[1])*(1+p[2]*(m/rs))
res = lambda p: (OBS-model_bkg(p,M))/ERR
params_input = [1,1.5,1]
fit  = leastsq(res,params_input)
popt = fit[0]
res0 = res(popt)
chi2bkg = np.dot(res0,res0)
chi2dof = chi2bkg/(OBS.size-popt.size)
print 'model for bkg'
print popt
print 'chi^2=',chi2bkg
print 'chi^2/dof=',chi2dof

check the results

In [ ]:
py.figure(figsize=(2*6,1*3))
ax = py.subplot(121)
ax.errorbar(M,OBS,yerr=ERR,fmt='r.',label=tex('bkg~data'))
ax.plot(M,model_bkg(popt,M),'k-',label=tex('bkg~fit'))
ax.set_ylabel(tex(r'OBS~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.legend(fontsize=20,frameon=0)

ax = py.subplot(122)
ax.errorbar(M,OBS,yerr=ERR,fmt='r.')
ax.plot(M,model_bkg(popt,M),'k-')
ax.set_ylabel(tex(r'OBS~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

# The signal

Retrieve simulated signal and construct lepton-pair invariant mass

In [ ]:
path = 'samples/sig.lhe.gz'
sig  = EVENTS(path)
sig.D['m'] = []
for i in range(sig.nevents):
    particles = sig.EVENTS[i]
    for p in particles:
        if p['pid']==-13: mub=p['mom']
        elif p['pid']==13: mu=p['mom']
    z = mu + mub
    sig.D['m'].append((z*z)**0.5) 

construct observable as $\frac{1}{N_{tot}}~~\frac{dN}{dm}$

In [ ]:
N,E = np.histogram(sig.D['m'],bins=number_of_bins,range=(lower_binning,upper_binning))

M    = 0.5*(E[:-1]+E[1:])
NTOT = np.sum(N)
SNSQ = np.sum(np.sqrt(N))
OBS  = N/float(NTOT)
ERR  = np.zeros(OBS.size)
for k in range(len(ERR)):
        ERR[k] = np.sqrt(N[k])/float(NTOT)

ax = py.subplot(111)
#plot_hist(ax,'r',E,OBS)
ax.errorbar(M,OBS,yerr=ERR,fmt='g.')
ax.set_ylabel(tex(r'sig~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

model the signal and make fit

In [ ]:
rs = 1300.0
model_sig = lambda p,m: p[0]*m**2 /((m**2-p[1]**2)**2+m**2 *p[2]**2)*(1+p[3]*(m/rs) + p[4]*(m/rs)**2)
res = lambda p: (OBS-model_sig(p,M))/ERR
params_input = np.zeros(5)+1
fit = leastsq(res,params_input)
popt = fit[0]
res0 = res(popt)
chi2sig = np.dot(res0,res0)
chi2dof = chi2sig/(OBS.size-popt.size)
print 'parameters=', popt
print 'chi^2=', chi2sig
print 'chi^2/dof=', chi2dof

check the results

In [ ]:
py.figure(figsize=(2*6,1*3))
ax = py.subplot(121)
ax.errorbar(M,OBS,yerr=ERR,fmt='r.',label=tex('sig~data'))
ax.plot(M,model_sig(popt,M),'k-',label=tex('sig~fit'))
ax.set_ylabel(tex(r'OBS~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.legend(fontsize=20,frameon=0)

ax = py.subplot(122)
ax.errorbar(M,OBS,yerr=ERR,fmt='r.')
ax.plot(M,model_sig(popt,M),'k-')
ax.set_ylabel(tex(r'OBS~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

# The signal + background

Retrieve simulated + background signal and construct lepton-pair invariant mass

In [ ]:
path = 'samples/sb.lhe.gz'
sb = EVENTS(path)
sb.D['m'] = []
for i in range(sb.nevents):
    particles = sb.EVENTS[i]
    for p in particles:
        if p['pid']==-13: mub=p['mom']
        elif p['pid']==13: mu=p['mom']
    z = mu + mub
    sb.D['m'].append((z*z)**0.5) 

construct observable as $\frac{1}{N_{tot}}~~\frac{dN}{dm}$

In [ ]:
N,E = np.histogram(sb.D['m'],bins=number_of_bins,range=(lower_binning,upper_binning))

M    = 0.5*(E[:-1]+E[1:])
NTOT = np.sum(N)
SNSQ = np.sum(np.sqrt(N))
OBS  = N/float(NTOT)
ERR  = np.zeros(OBS.size)
for k in range(len(ERR)):
        ERR[k] = np.sqrt(N[k])/float(NTOT)

ax = py.subplot(111)
ax.errorbar(M,OBS,yerr=ERR,fmt='g.')
ax.set_ylabel(tex(r'sig+bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();

model the signal+background and make fit

In [ ]:
rs = 1300.0
model_sig = lambda p,m: p[0]*m**2 /((m**2-p[1]**2)**2+m**2 *p[2]**2)*(1+p[3]*(m/rs) + p[4]*(m/rs)**2)
model_bkg = lambda p,m: p[5]/m**(p[6])
model_sb  = lambda p,m: model_sig(p,m) + model_bkg(p,m)
res = lambda p: (OBS-model_sb(p,M))/ERR
params_input = [1,91,1,1,1,1,1]
fit = leastsq(res,params_input,full_output=1)
popt = fit[0]
res0 = res(popt)
chi2sb  = np.dot(res0,res0)
chi2dof = chi2sb/(OBS.size-popt.size)
print 'parameters=', popt
print 'chi^2=', chi2sb
print 'chi^2/dof=', chi2dof

In [ ]:
py.figure(figsize=(2*6,2*4))
ax = py.subplot(221)
ax.errorbar(M,OBS,yerr=ERR,fmt='r.',label=tex('sig+bkg~data'))
ax.plot(M,model_sb(popt,M),'k-',label=tex('sig+bkg~fit'))
ax.set_ylabel(tex(r'sig+bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.semilogy();
ax.legend(fontsize=15,frameon=0)

ax = py.subplot(222)
m0_lower_limit = 80
m0_upper_limit = 100
m0_step = 200
ax.set_xlim(m0_lower_limit,m0_upper_limit)
M0 = np.linspace(m0_lower_limit,m0_upper_limit,m0_step)
ax.plot(M0,model_sb(popt,M0),'k-',label=tex('sig+bkg~fit'))
ax.errorbar(M,OBS,yerr=ERR,fmt='r.',label=tex('sig+bkg~data'))
ax.set_ylabel(tex(r'sig+bkg~(a.u)'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)

ax = py.subplot(223)
ax.errorbar(M,OBS/model_sb(popt,M),yerr=ERR/model_sb(popt,M),fmt='r.',label=tex('sig+bkg~data'))
ax.set_ylabel(tex(r'D/T'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.axhline(1)
#ax.semilogy();

CHI2 = (OBS-model_sb(popt,M))/ERR
ax = py.subplot(224)
ax.plot(M,CHI2,'r.')
ax.set_ylabel(tex(r'\chi^2_i'),size=20)
ax.set_xlabel(tex(r'm (GeV)'),size=20)
ax.axhline(0);

# Correlation plot

In [ ]:
popt, pcov, infodict, errmsg, ier = fit
fjac = infodict['fjac']
npts = OBS.size
s_sq = (infodict['fvec']**2).sum()/npts
cov  = pcov * s_sq
cor  = np.zeros(cov.shape)

I = len(cov)
for i in range(I):
    for j in range(I):
        cor[i,j] = cov[i,j]/np.sqrt(cov[i,i]*cov[j,j])

ax = py.subplot(111)
pc = ax.pcolor(cor,cmap='bwr')

s = [[str(e) for e in row] for row in cor]
lens = [max(map(len, col)) for col in zip(*s)]
fmt = '\t'.join('{{:{}}}'.format(x) for x in lens)
table = [fmt.format(*row) for row in s]
#print '\n'.join(table)

ax.set_xticks(np.arange(len(cor))+0.5)
ax.set_xticklabels(['p%d'%i for i in range(len(cor))],rotation=90)
ax.set_yticks(np.arange(len(cor))+0.5)
ax.set_yticklabels(['p%d'%i for i in range(len(cor))],rotation=90)
ax.set_xlim(0,len(cor))
ax.set_ylim(0,len(cor))

py.colorbar(pc);
